In [3]:
## 1. ////////////////////////////////////libraries/////////////////////////////

import numpy as np
import gmsh
import sys
import os
sys.path.append(os.path.abspath("../.."))

import Geo2Gmsh


## 2. ///////////////////////////////////////start///////////////////////////////
    
gmsh.initialize()
gmsh.clear() 
gmsh.model.add("Overview_example")
    
# Here, we define a scalar factor and apply it to all functions. This line can be deleted or modified.
exag = 1

# Global counters. DO NOT DELETE!
Geo2Gmsh.counter = 0
Geo2Gmsh.counter1 = 0

## 3. ///////////// functions and mesh settings //////////////////
    
Geo2Gmsh.create_surface(
    surf_id=1,
    file_name="layer_1.txt",
    samx=448,
    samy=448,
    v_ex=1,
    show_color=False
)
    
Geo2Gmsh.create_surface(
    surf_id=2,
    file_name="layer_2.txt",
    samx=448,
    samy=448,
    v_ex=1,
    show_color=False
)
    
volumes = Geo2Gmsh.volume_generation(
    num_loaded_surfaces = 2
)
    
fault_1 = Geo2Gmsh.add_fault(
    file_name="fault_1.txt",
    v_ex=1,
    surf_id = 1,
    fault_id = 1,
    dip = 65.,
    dip_dir = 280.,
    fault_len = 5.
)
 
well_1 = Geo2Gmsh.add_well(
    file_name="well_1.txt",
    v_ex=1,
    well_id=1
)
   
Geo2Gmsh.local_refinement(
    element_type = "well",
    element_list = well_1,
    sampling =100,
    Size_Min =0.5,
    Size_Max =2.0,
    Dist_Min =3,
    Dist_Max =5
)
    
    
Geo2Gmsh.local_refinement(
    element_type = "fault",
    element_list = fault_1,
    sampling =100,
    Size_Min = 0.8,
    Size_Max = 4.,
    Dist_Min = 3,
    Dist_Max = 5
)
    
Geo2Gmsh.local_refinement(
    element_type = "surface",
    element_list = [2],
    sampling =1000,
    Size_Min = 1.2,
    Size_Max = 4,
    Dist_Min = 2.,
    Dist_Max = 3.
)
    
    
## 4. ////////  refinement field setings /////////// 
Geo2Gmsh.counter += 1
gmsh.model.mesh.field.add("Min", Geo2Gmsh.counter)
gmsh.model.mesh.field.setNumbers(Geo2Gmsh.counter, "FieldsList", [field for field in range(2, Geo2Gmsh.counter, 2)])
gmsh.model.mesh.field.setAsBackgroundMesh(Geo2Gmsh.counter)
    
    
##/////////////////////Global refinement////////////////////////// 
gmsh.option.setNumber("Mesh.MeshSizeMin",0.4)
gmsh.option.setNumber("Mesh.MeshSizeMax", 4. )
# For certain errors changing the meshing algoritms can help. In this case algor. 2. (see Gmsh documentation)
gmsh.option.setNumber("Mesh.Algorithm3D", 2)  
gmsh.option.setNumber("Geometry.Tolerance", 0.4) #This allows Gmsh recognize close points (distance>0.4) as different nodes. If this option is properly configured, geometries such as wedge layering can be represented
    
## 5. ////////////////////physical group assignment////////////////////////////
    
# Geo2Gmsh.physical_group(
#     element_type = "well",
#     element_list = well_1
# )
    
# Geo2Gmsh.physical_group(
#     element_type = "fault",
#     element_list = fault_1
# )
    
# Geo2Gmsh.physical_group(
#     element_type = "surface",
#     element_list = [2]
# )
    
Geo2Gmsh.physical_group(
     element_type = "volume",
     element_list = volumes)
    
## 6. ////////////////////meshing////////////////////////////     
gmsh.model.geo.synchronize() #Super important. Not delete this line!!
gmsh.model.mesh.generate(3) #Super important. Not delete this line!!
    
#///////// visulization (optional) ////////////////////
gmsh.option.setNumber("Mesh.SurfaceEdges", 1)
#gmsh.option.setNumber("Mesh.VolumeEdges", 1)
#gmsh.option.setNumber("Mesh.SurfaceFaces", 1)
gmsh.option.setNumber("Mesh.VolumeFaces", 1)  
    
## 7. ///////// output format ////////////
gmsh.write("outputs/overview.msh")
gmsh.write("outputs/overview.vtk")
      
#///// if working with sfepy the original vtk must be modified////
with open("outputs/overview.vtk", "r") as f:
    lines = f.readlines()
    
with open("outputs/overview.vtk", "w") as f:
    for line in lines:
        if "SCALARS CellEntityIds" in line:
            line = "SCALARS mat_id long\n"
        f.write(line)
    
gmsh.fltk.run()
gmsh.clear() 
gmsh.finalize()


Superficie 1 successfully created.
Superficie 2 successfully created.
Volumes 1 succesfully created.
Fault 1 succesfully created.
Well 1 succesfully created.
Mesh succesfully refined around wells.
Mesh succesfully refined around faults.
Mesh succesfully refined around surface [2].
physical_id_(volume 1): 1
